# Halide Perovskite X-ray + UV-Vis — Single Trial Pre-Flight Check

Submits **one acquisition trial** through the full queue-server stack to verify
the end-to-end path (pumps → QEPro → pe1c → evaluation) before committing to a
full X-ray + UV-Vis campaign.

## ⚠️ Physical safety — read before running

> **Disconnect the DDS and Ultra syringes from their drive screws before
> running this notebook.**  The pump motors will receive a move command and
> the screws will turn.  Without disconnection you will dispense fluid.

## Prerequisites

1. `validate_paths.ipynb` ran cleanly — all devices connected (including `pe1c`).

2. Queue-server worker running with `97-xray_uvvis.py` + `98-hack.py` deployed.

3. A saved campaign config JSON produced by `halide_beamline_xray_uvvis.ipynb`
   (e.g. `configs/halide_beamline_xray_uvvis.json`).  Set `XPD_HALIDE_CONFIG`
   in your `.env` or shell to point at it.

## Bring-up sequence

| Step | Notebook / command |
|------|-------------------|
| 1 | `validate_paths.ipynb` — device & PV connectivity (read-only) |
| 2 | `single_trial_xray_uvvis.ipynb` — **this notebook** — one acquisition trial |
| 3 | `halide_beamline_xray_uvvis.ipynb` — full optimization campaign |

---
## 1 — Environment

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

env_keys = [
    "QSERVER_HTTP_URI",
    "QSERVER_HTTP_SERVER_API_KEY",
    "ZMQ_CONSUMER_ADDR",
    "TILED_URI",
    "XPD_HALIDE_CONFIG",
]

print("Environment variables:")
print("-" * 50)
for key in env_keys:
    val = os.environ.get(key, "<not set>")
    if "KEY" in key and val != "<not set>":
        val = val[:6] + "..." if len(val) > 6 else "***"
    print(f"  {key:<35} {val}")

CONFIG_PATH: str = os.environ.get(
    "XPD_HALIDE_CONFIG", "configs/halide_beamline_xray_uvvis.json"
)
print()
print(f"  Config path  : {CONFIG_PATH}")
print(f"  Config exists: {Path(CONFIG_PATH).exists()}")

---
## 2 — Load campaign config

Reconstructs the `BuildAgent` from the archived JSON so the single trial
uses exactly the same experiment parameters as the campaign.

In [ ]:
from xpd_tools.optimization.campaign import load_campaign_config

build_agent = load_campaign_config(CONFIG_PATH)

print("BuildAgent reconstructed:")
print(f"  evaluation_method  : {build_agent.evaluation_method}")
print(f"  DOFs               : {[p.id for p in build_agent.pumps]}")
print(f"  rate_unit          : {build_agent.rate_unit}")
print(f"  mixer_lengths_cm   : {build_agent.mixer_lengths_cm}")
print(f"  residence_time_ratio: {build_agent.residence_time_ratio}")
print(f"  tubing_id_mm       : {build_agent.tubing_id_mm}")
if build_agent.xray_settings is not None:
    print(f"  xray.exposure      : {build_agent.xray_settings.exposure} s")
    print(f"  xray.stream_name   : {build_agent.xray_settings.stream_name}")

---
## 3 — Build agent and verify plan registration

In [ ]:
from xpd_tools.optimization.campaign import assert_plan_registered

run_agent = build_agent.build()   # -> QueueserverAgent

print("QueueserverAgent built.")
print(f"  acquisition_plan : {run_agent.acquisition_plan}")
print(f"  actuators        : {list(run_agent.actuators)}")
print()

assert_plan_registered(run_agent, plan_name="xray_uvvis_acquire")

---
## 4 — Define trial point

Adjust rates (ul/min) to match your current reagent setup.
Every rate must fall within its pump's min/max range for the loaded syringe.

| DOF key | Device | Reagent |
|---------|--------|---------|
| `infusion_rate_CsPb`  | `dds2_p1` | CsPb precursor |
| `infusion_rate_TOABr` | `dds2_p2` | TOABr |
| `infusion_rate_Cl`    | `dds3_p1` | Pb-rich / TOACl |
| `infusion_rate_I`     | `dds3_p2` | TOAI |

In [ ]:
# Rates in ul/min — adjust to your reagent setup.
single_trial_point: list[dict[str, float]] = [{
    "infusion_rate_CsPb":  40.0,   # dds2_p1 — CsPb precursor
    "infusion_rate_TOABr": 30.0,   # dds2_p2 — TOABr
    "infusion_rate_Cl":    10.0,   # dds3_p1 — Pb-rich / TOACl
    "infusion_rate_I":     20.0,   # dds3_p2 — TOAI
}]

print("Trial point:")
for k, v in single_trial_point[0].items():
    print(f"  {k:<25} {v} ul/min")

---
## 5 — Submit and block

`submit_suggestions` sends the point to the queue server and returns a
`Future`.  `.result()` blocks until the acquisition and evaluation are done.

**What to watch:**
- Drive screws begin turning within a few seconds of submission.
- Worker output shows dark/reference/sample spectrum capture, then X-ray exposure.
- A failed peak fit on a dark frame is expected — the acquisition ran but found
  no PL signal (correct for a dark frame).

In [ ]:
print("Submitting trial to queue server...")
future = run_agent.submit_suggestions(single_trial_point)

print("Waiting for acquisition and evaluation to complete...")
result = future.result()   # blocks until done

print("\nResult:")
print(result)

---
## Next Steps

| Outcome | Next action |
|---------|-------------|
| Screws turned, result returned | Reconnect syringes and open `halide_beamline_xray_uvvis.ipynb` |
| Screws did not turn | Check pump EPICS IOC, DDS comms state (`Enabled`), and pump rate bounds |
| UV-Vis evaluation error | Inspect worker logs; a dark-frame fit failure is expected and harmless |
| X-ray evaluation error | Check pe1c connectivity; verify `periodic_dark_02` loaded in worker |